# 🗑️ SPIRO ML — Full Training Pipeline
## Google Colab Free Tier (T4 GPU)

**This notebook trains SPIRO waste detection end-to-end:**
1. Install dependencies
2. Download & prepare datasets (TACO, ZeroWaste-f, TrashNet, Kaggle GC)
3. Map to SPIRO 109-class taxonomy
4. Train YOLOv11n (nano — fits free T4)
5. Export to ONNX
6. Train EfficientNetV2-B0 verifier
7. Export verifier to ONNX
8. Package everything for download

**⚠️ Important:** Enable GPU first: Runtime → Change runtime type → T4 GPU

**⏱️ Estimated time:** ~3-4 hours on free T4

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else '❌ No GPU! Enable GPU in Runtime settings.')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem:.1f} GB')

In [ ]:
# ── Cell 2: Mount Google Drive (for saving checkpoints) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/SPIRO_ML'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Checkpoints will save to: {SAVE_DIR}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics==8.2.0 -q
!pip install timm==1.0.3 -q
!pip install onnx==1.16.0 onnxruntime==1.18.1 onnxsim -q
!pip install albumentations -q
!pip install roboflow -q
!pip install pycocotools -q
!pip install omegaconf -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 4: Upload SPIRO package & extract ────────────────────────────────────
# Upload spiro_colab.zip using the Files panel on the left
# OR run this cell to upload interactively:
from google.colab import files
print('Upload spiro_colab.zip (the file you downloaded from Claude):')
uploaded = files.upload()

In [ ]:
# ── Cell 5: Extract package ───────────────────────────────────────────────────
import zipfile, os

# Find the zip
zip_file = next((f for f in os.listdir('.') if f.endswith('.zip')), None)
if zip_file:
    with zipfile.ZipFile(zip_file) as z:
        z.extractall('/content/spiro')
    print(f'✅ Extracted: {zip_file}')
else:
    print('❌ No zip found — make sure you uploaded spiro_colab.zip')

os.chdir('/content/spiro')
!ls

In [ ]:
# ── Cell 6: Run full setup & dataset download ─────────────────────────────────
# This downloads all datasets, maps to 109 SPIRO classes, creates YOLO splits
!python scripts/setup_datasets.py --all 2>&1 | tail -50

In [ ]:
# ── Cell 7: Show dataset statistics ──────────────────────────────────────────
!python scripts/dataset_stats.py

In [ ]:
# ── Cell 8: Train YOLOv11n ────────────────────────────────────────────────────
# nano model — designed for free T4 (15GB VRAM)
!python scripts/train_yolo.py \
    --variant nano \
    --epochs 100 \
    --save-dir /content/drive/MyDrive/SPIRO_ML \
    2>&1 | tail -80

In [ ]:
# ── Cell 9: Export YOLOv11 to ONNX ───────────────────────────────────────────
!python scripts/export_yolo.py \
    --weights runs/detect/spiro_yolov11n/weights/best.pt \
    --save-dir /content/drive/MyDrive/SPIRO_ML

In [ ]:
# ── Cell 10: Train EfficientNetV2-B0 verifier ─────────────────────────────────
!python scripts/train_effnet.py \
    --variant b0 \
    --epochs 30 \
    --save-dir /content/drive/MyDrive/SPIRO_ML \
    2>&1 | tail -60

In [ ]:
# ── Cell 11: Export EfficientNetV2 to ONNX ────────────────────────────────────
!python scripts/export_effnet.py \
    --weights runs/verify/spiro_effnetv2_b0/best_effnet.pt \
    --save-dir /content/drive/MyDrive/SPIRO_ML

In [ ]:
# ── Cell 12: Run test inference ───────────────────────────────────────────────
!python scripts/test_inference.py \
    --yolo /content/drive/MyDrive/SPIRO_ML/yolo_best.onnx \
    --effnet /content/drive/MyDrive/SPIRO_ML/effnet_best.onnx

In [ ]:
# ── Cell 13: Package final models ─────────────────────────────────────────────
!python scripts/package_models.py --save-dir /content/drive/MyDrive/SPIRO_ML
print('\n✅ Training complete! Download spiro_models.zip from your Google Drive.')